# Day 1: Predicting COX-1 and COX-2 Structures from Sequence Alone
**Project:** Can AI predict how drugs bind? Benchmarking structure and co-folding models on COX enzymes.

Today we ask the foundation question: **if we give an AI model only the amino acid sequence, can it recover the real 3D structure, including the drug-binding pocket?**

We use **ESMFold**, a protein language model that predicts structure in a single pass, and compare it against experimental crystal structures:
- COX-1 (PDB 1EQG, ibuprofen bound)
- COX-2 (PDB 3LN1, celecoxib bound)

**What we measure**
1. Global accuracy: RMSD, TM-score, GDT (approximate)
2. Confidence: pLDDT per residue, and whether it predicts where errors are
3. Pocket accuracy: the residues around the drug, including the selectivity-determining positions 513 and 523

**Use a GPU runtime** (Runtime → Change runtime type → T4 GPU).

## 1. Setup

In [ ]:
!pip install -q transformers accelerate biopython py3Dmol pandas matplotlib
import torch
print("GPU available:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
import os, requests
import numpy as np, pandas as pd, matplotlib.pyplot as plt, py3Dmol
from Bio import Align
from Bio.Align import substitution_matrices
from Bio.PDB import PDBParser, PDBIO, Superimposer

THREE2ONE = {"ALA":"A","ARG":"R","ASN":"N","ASP":"D","CYS":"C","GLN":"Q","GLU":"E","GLY":"G","HIS":"H",
             "ILE":"I","LEU":"L","LYS":"K","MET":"M","PHE":"F","PRO":"P","SER":"S","THR":"T","TRP":"W",
             "TYR":"Y","VAL":"V","MSE":"M"}
SKIP = {"HOH","HEM","NAG","BMA","MAN","FUC","GOL","SO4","EDO","BOG","NA","CA","CL"}

def xyz(ls):
    return np.array([[float(l[30:38]), float(l[38:46]), float(l[46:54])] for l in ls])

## 2. Load the experimental structures and their sequences
We take each sequence directly from the PDB entry (SEQRES records), so the model predicts exactly the protein that was crystallized.

In [ ]:
def load_experimental(key, pdb_id, prefer):
    text = requests.get(f"https://files.rcsb.org/download/{pdb_id}.pdb", timeout=30).text
    lines = text.splitlines()
    groups = {}
    for l in lines:
        if l.startswith("HETATM") and l[16] in " A" and l[17:20].strip() not in SKIP:
            groups.setdefault((l[17:20].strip(), l[21]), []).append(l)
    choice = next((k for p in prefer for k in groups if k[0] == p), None) or max(groups, key=lambda k: len(groups[k]))
    lig_name, chain_id = choice
    lig = groups[choice]
    seq = "".join(THREE2ONE.get(r, "X") for l in lines if l.startswith("SEQRES") and l[11] == chain_id
                  for r in l[19:].split())
    atoms = [l for l in lines if l.startswith("ATOM") and l[21] == chain_id and l[16] in " A"]
    open(f"exp_{key}.pdb", "w").write("\n".join(atoms) + "\nEND\n")
    open(f"exp_{key}_view.pdb", "w").write("\n".join(atoms + lig) + "\nEND\n")
    s = PDBParser(QUIET=True).get_structure(key, f"exp_{key}.pdb")
    res = [r for r in s[0][chain_id] if r.id[0] == " " and "CA" in r]
    return {"key": key, "pdb_id": pdb_id, "chain": chain_id, "lig_name": lig_name, "lig": lig,
            "seq": seq, "res": res, "struct": s}

EXP = {
    "cox1": load_experimental("cox1", "1EQG", ["IBP"]),
    "cox2": load_experimental("cox2", "3LN1", ["CEL", "S58"]),
}
for k, e in EXP.items():
    print(f"{k}: PDB {e['pdb_id']} chain {e['chain']} | ligand {e['lig_name']} | "
          f"sequence length {len(e['seq'])} | resolved residues {len(e['res'])}")

In [ ]:
# How similar are the two sequences?
blosum = Align.PairwiseAligner()
blosum.substitution_matrix = substitution_matrices.load("BLOSUM62")
blosum.open_gap_score, blosum.extend_gap_score = -10, -0.5
aln = blosum.align(EXP["cox1"]["seq"], EXP["cox2"]["seq"])[0]
matches = sum(a == b for a, b in zip(*[str(x) for x in (aln[0], aln[1])]) if a != "-" and b != "-")
cols = sum(1 for a, b in zip(str(aln[0]), str(aln[1])) if a != "-" and b != "-")
print(f"COX-1 vs COX-2 sequence identity over aligned positions: {100*matches/cols:.0f}%")

## 3. Load ESMFold
The first run downloads several GB of weights and takes a few minutes. The language model part runs in half precision and the folding trunk runs in chunks to fit in GPU memory.

In [ ]:
from transformers import EsmForProteinFolding
torch.backends.cuda.matmul.allow_tf32 = True
model = EsmForProteinFolding.from_pretrained("facebook/esmfold_v1", low_cpu_mem_usage=True)
model = model.cuda().eval()
model.esm = model.esm.half()
print("ESMFold loaded")

## 4. Fold both sequences
Expect roughly 1-3 minutes per protein on a T4. If memory runs out, the code retries with smaller chunks automatically.

In [ ]:
def fold(seq, name):
    path = f"pred_{name}.pdb"
    if os.path.exists(path):
        return path
    for chunk in (64, 32, 16):
        try:
            model.trunk.set_chunk_size(chunk)
            with torch.no_grad():
                pdb_str = model.infer_pdb(seq)
            break
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            print(f"Out of memory at chunk {chunk}, retrying smaller")
    else:
        raise RuntimeError("Could not fit this sequence on the GPU")
    open(path, "w").write(pdb_str)
    return path

PRED_PATH = {k: fold(e["seq"], k) for k, e in EXP.items()}
print(PRED_PATH)

In [ ]:
def load_pred(key):
    s = PDBParser(QUIET=True).get_structure(key, PRED_PATH[key])
    atoms = list(s.get_atoms())
    if max(a.get_bfactor() for a in atoms) <= 1.0:      # make pLDDT 0-100
        for a in atoms:
            a.set_bfactor(a.get_bfactor() * 100)
    return s

PRED = {k: load_pred(k) for k in EXP}
for k, s in PRED.items():
    pl = [r["CA"].get_bfactor() for r in next(s[0].get_chains()) if "CA" in r]
    print(f"{k}: mean pLDDT {np.mean(pl):.1f} | residues with pLDDT > 90: {100*np.mean(np.array(pl) > 90):.0f}%")

## 5. Compare prediction to experiment
Steps: align sequences to match residues, superimpose on **confident** residues only (pLDDT > 70), then measure the distance between matching C-alpha atoms.

TM-score and GDT here are *approximate*: they use one fixed superposition instead of the optimized search done by dedicated tools.

In [ ]:
aligner = Align.PairwiseAligner()
aligner.mode = "global"
aligner.match_score, aligner.mismatch_score = 2, -1
aligner.open_gap_score, aligner.extend_gap_score = -5, -0.5

def seq_of(residues):
    return "".join(THREE2ONE.get(r.get_resname(), "X") for r in residues)

def map_residues(pred_seq, exp_seq):
    aln = aligner.align(pred_seq, exp_seq)[0]
    pairs = []
    for (p0, p1), (e0, e1) in zip(*aln.aligned):
        pairs += list(zip(range(p0, p1), range(e0, e1)))
    return pairs

def compare(key):
    info, pred = EXP[key], PRED[key]
    exp_res = info["res"]
    pred_res = [r for r in next(pred[0].get_chains()) if "CA" in r]
    pairs = map_residues(seq_of(pred_res), seq_of(exp_res))
    plddt = np.array([pred_res[p]["CA"].get_bfactor() for p, _ in pairs])
    fixed = [exp_res[e]["CA"] for _, e in pairs]
    moving = [pred_res[p]["CA"] for p, _ in pairs]
    core = plddt > 70
    sup = Superimposer()
    sup.set_atoms([f for f, c in zip(fixed, core) if c], [m for m, c in zip(moving, core) if c])
    sup.apply(list(pred[0].get_atoms()))
    d = np.linalg.norm(np.array([f.coord for f in fixed]) - np.array([m.coord for m in moving]), axis=1)
    L = len(exp_res)
    d0 = 1.24 * (L - 15) ** (1 / 3) - 1.8
    io = PDBIO(); io.set_structure(pred); io.save(f"pred_{key}_aligned.pdb")
    return {"pairs": pairs, "pred_res": pred_res, "plddt": plddt, "d": d, "core": core,
            "rmsd_all": float(np.sqrt((d ** 2).mean())),
            "rmsd_core": float(np.sqrt((d[core] ** 2).mean())),
            "tm": float(np.sum(1 / (1 + (d / d0) ** 2)) / L),
            "gdt": float(np.mean([(d < c).sum() / L * 100 for c in (1, 2, 4, 8)])),
            "within2": float((d < 2).mean() * 100),
            "mean_plddt": float(np.mean([r["CA"].get_bfactor() for r in pred_res]))}

CMP = {k: compare(k) for k in EXP}
for k, c in CMP.items():
    print(f"{k}: RMSD all {c['rmsd_all']:.2f} Å | RMSD confident {c['rmsd_core']:.2f} Å | "
          f"TM≈{c['tm']:.2f} | GDT≈{c['gdt']:.0f} | within 2 Å: {c['within2']:.0f}%")

## 6. Where are the errors? Does confidence predict them?

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6.5))
for ax, (k, c) in zip(axes, CMP.items()):
    resnum = [EXP[k]["res"][e].id[1] for _, e in c["pairs"]]
    ax.bar(resnum, np.clip(c["d"], 0, 15), color="#E45756", width=1.0)
    ax.set_ylabel("C-alpha deviation (Å)"); ax.set_title(f"{k.upper()}: deviation (bars) and pLDDT (line)")
    ax2 = ax.twinx(); ax2.plot(resnum, c["plddt"], color="#4C78A8", lw=1); ax2.set_ylabel("pLDDT"); ax2.set_ylim(0, 100)
axes[1].set_xlabel("Residue number (experimental numbering)")
plt.tight_layout(); plt.savefig("day1_deviation_plddt.png", dpi=150); plt.show()

for k, c in CMP.items():
    r = np.corrcoef(c["plddt"], c["d"])[0, 1]
    print(f"{k}: correlation between pLDDT and error = {r:.2f} (strongly negative means confidence flags the errors)")

## 7. Pocket accuracy: the part that matters for drugs
We take every residue with an atom within 6 Å of the crystal inhibitor and compare side-chain atoms of the prediction to the experiment (after the global superposition). Side-chain RMSD is only computed where the predicted residue type matches.

In [ ]:
KEY_RES = (120, 355, 385, 513, 523, 530)

def pocket_analysis(key, cutoff=6.0):
    info, c = EXP[key], CMP[key]
    exp_res, pred_res = info["res"], c["pred_res"]
    lig_xyz = xyz(info["lig"])
    pair_map = {e: p for p, e in c["pairs"]}
    rows = []
    for ei, r in enumerate(exp_res):
        heavy = np.array([a.coord for a in r if not a.get_id().startswith("H")])
        in_pocket = np.linalg.norm(heavy[:, None, :] - lig_xyz[None, :, :], axis=2).min() < cutoff
        if not (in_pocket or r.id[1] in KEY_RES) or ei not in pair_map:
            continue
        pr = pred_res[pair_map[ei]]
        ca_dev = float(np.linalg.norm(r["CA"].coord - pr["CA"].coord))
        sc = None
        if r.get_resname() == pr.get_resname():
            names = [a.get_id() for a in r if a.get_id() in pr and a.get_id() not in ("N", "CA", "C", "O")
                     and not a.get_id().startswith("H")]
            if names:
                sc = float(np.sqrt(np.mean([np.sum((r[n].coord - pr[n].coord) ** 2) for n in names])))
        rows.append({"Residue": f"{r.get_resname()}{r.id[1]}", "Predicted as": pr.get_resname(),
                     "pLDDT": round(pr["CA"].get_bfactor(), 1), "CA dev (Å)": round(ca_dev, 2),
                     "Side-chain RMSD (Å)": None if sc is None else round(sc, 2),
                     "in_pocket": in_pocket, "resseq": r.id[1]})
    return pd.DataFrame(rows)

POCKET = {k: pocket_analysis(k) for k in EXP}
for k, df in POCKET.items():
    p = df[df.in_pocket]
    print(f"\n{k.upper()}: {len(p)} pocket residues | mean pLDDT {p['pLDDT'].mean():.1f} | "
          f"mean CA dev {p['CA dev (Å)'].mean():.2f} Å | mean side-chain RMSD {p['Side-chain RMSD (Å)'].mean():.2f} Å")
    display(df[df.resseq.isin(KEY_RES)].drop(columns=["in_pocket", "resseq"]).reset_index(drop=True))

**Reading the key residues:** Arg120, Tyr355, Tyr385, and Ser530 are the classic binding-site residues. Residue 523 should be isoleucine in COX-1 and valine in COX-2, and residue 513 should be histidine in COX-1 and arginine in COX-2. If the numbering in the table does not match this, the PDB entry uses different numbering, so rely on the residue names you see.

## 8. 3D comparison
Gray: experimental structure · rainbow: prediction colored by confidence (blue high, red low) · magenta: crystal inhibitor · orange sticks: positions 513 and 523.

In [ ]:
def show(key):
    info, c = EXP[key], CMP[key]
    pair_map = {e: p for p, e in c["pairs"]}
    exp_ids = [r.id[1] for r in info["res"] if r.id[1] in (513, 523)]
    pred_ids = [c["pred_res"][pair_map[i]].id[1] for i, r in enumerate(info["res"])
                if r.id[1] in (513, 523) and i in pair_map]
    v = py3Dmol.view(width=600, height=480)
    v.addModel(open(f"exp_{key}_view.pdb").read(), "pdb")
    v.setStyle({"model": 0}, {"cartoon": {"color": "lightgray", "opacity": 0.7}})
    v.setStyle({"model": 0, "hetflag": True}, {"stick": {"colorscheme": "magentaCarbon"}})
    v.addStyle({"model": 0, "resi": exp_ids}, {"stick": {"colorscheme": "orangeCarbon", "radius": 0.2}})
    v.addModel(open(f"pred_{key}_aligned.pdb").read(), "pdb")
    v.setStyle({"model": 1}, {"cartoon": {"colorscheme": {"prop": "b", "gradient": "roygb", "min": 50, "max": 90}}})
    v.addStyle({"model": 1, "resi": pred_ids}, {"stick": {"colorscheme": "orangeCarbon", "radius": 0.2}})
    v.zoomTo({"model": 0, "hetflag": True})
    return v

show("cox2").show()

In [ ]:
show("cox1").show()

## 9. Summary table for the README

In [ ]:
rows = []
for k in EXP:
    c, p = CMP[k], POCKET[k]; pp = p[p.in_pocket]
    rows.append({"Enzyme": k.upper(), "PDB": EXP[k]["pdb_id"], "Length": len(EXP[k]["seq"]),
                 "Mean pLDDT": round(c["mean_plddt"], 1),
                 "RMSD all (Å)": round(c["rmsd_all"], 2), "RMSD confident (Å)": round(c["rmsd_core"], 2),
                 "TM-score≈": round(c["tm"], 2), "GDT≈": round(c["gdt"], 0),
                 "Pocket CA dev (Å)": round(pp["CA dev (Å)"].mean(), 2),
                 "Pocket side-chain RMSD (Å)": round(pp["Side-chain RMSD (Å)"].mean(), 2)})
summary = pd.DataFrame(rows)
summary.to_csv("day1_summary.csv", index=False)
cols = list(summary.columns)
print("| " + " | ".join(cols) + " |")
print("|" + "---|" * len(cols))
for _, r in summary.iterrows():
    print("| " + " | ".join(str(r[c]) for c in cols) + " |")

## What to write in your README (after you see the numbers)
- Did ESMFold recover the global fold? (TM-score above 0.5 means the same fold; above 0.9 is very close.)
- Did confidence (pLDDT) flag the regions that were wrong?
- Was the pocket as accurate as the whole protein, or worse? Side-chain positions in the pocket matter most for docking.
- Did the model get the COX-1 vs COX-2 differences at 513 and 523 right?

## Limitations
- ESMFold predicts the **apo** protein: no drug, no heme cofactor, single chain.
- One crystal structure is one snapshot; proteins move, so some "error" is real flexibility.
- TM-score and GDT here are approximations using a single superposition.
- COX-1 is the ovine enzyme and COX-2 the mouse enzyme (standard crystal models).

## Tomorrow (Day 2)
Protein language model embeddings (ESM-2): do they capture what makes COX-2 different from COX-1?